# Modelado Predictivo — Predicción de Churn

**Asignatura:** Machine Learning  
**Algoritmos:** Random Forest · XGBoost · CatBoost · LightGBM  
**Objetivo:** Entrenar y comparar modelos con Pipeline y GridSearchCV para seleccionar el mejor clasificador.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import joblib
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

Se importan scikit-learn para los pipelines y métricas, XGBoost, CatBoost y LightGBM como clasificadores, y `joblib` y `json` para serializar el modelo y sus metadatos. Los warnings se suprimen para mantener la salida limpia durante el GridSearchCV.

In [ ]:
# Curvas ROC comparativas en una sola figura
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, roc_auc_score

fig, ax = plt.subplots(figsize=(9, 7))

colors = {
    "Random Forest": "#e74c3c",
    "XGBoost": "#27ae60",
    "CatBoost": "#f39c12",
    "LightGBM": "#8e44ad"
}

for model_name, m in best_estimators.items():
    y_proba_m = m.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba_m)
    RocCurveDisplay.from_predictions(
        y_test, y_proba_m,
        name=f"{model_name}  (AUC = {auc:.4f})",
        ax=ax, color=colors.get(model_name, "blue"),
        linewidth=2
    )

ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Clasificador Aleatorio (AUC = 0.50)")
ax.set_title(
    "Curvas ROC — Comparación de Modelos\n(Conjunto de Prueba, n=1,409)",
    fontsize=14, fontweight="bold"
)
ax.set_xlabel("Tasa de Falsos Positivos (1 - Especificidad)", fontsize=12)
ax.set_ylabel("Tasa de Verdaderos Positivos (Sensibilidad)", fontsize=12)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()


Los cuatro modelos tienen curvas muy cercanas entre sí, todas bien por encima de la diagonal de un clasificador aleatorio. El AUC varía entre 0.839 y 0.848, lo que indica que todos capturan los patrones de churn de forma similar. XGBoost (0.8481) queda levemente más arriba que los demás.

In [ ]:
# Matrices de confusión para todos los modelos (seaborn heatmap)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

model_order = ["Random Forest", "XGBoost", "CatBoost", "LightGBM"]

for i, model_name in enumerate(model_order):
    if model_name not in best_estimators:
        continue
    m = best_estimators[model_name]
    y_pred_m = m.predict(X_test)
    cm = confusion_matrix(y_test, y_pred_m)

    # Calcular porcentajes
    cm_pct = cm / cm.sum() * 100
    labels = [[f"{v}\n({p:.1f}%)" for v, p in zip(row_v, row_p)]
              for row_v, row_p in zip(cm, cm_pct)]

    color = {"Random Forest": "Blues", "XGBoost": "Greens",
             "CatBoost": "Oranges", "LightGBM": "Purples"}.get(model_name, "Blues")

    sns.heatmap(
        cm, annot=labels, fmt="", cmap=color,
        xticklabels=["No Churn", "Churn"],
        yticklabels=["No Churn", "Churn"],
        ax=axes[i], linewidths=0.5, linecolor="gray",
        cbar_kws={"shrink": 0.8}
    )
    axes[i].set_title(f"{model_name}", fontsize=14, fontweight="bold", pad=10)
    axes[i].set_xlabel("Predicción", fontsize=11)
    axes[i].set_ylabel("Valor Real", fontsize=11)

plt.suptitle(
    "Matrices de Confusión — Comparación de Modelos\n(Conjunto de Prueba, n=1,409)",
    fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()


Las matrices muestran que los cuatro modelos clasifican bien la clase "No Churn" —los verdaderos negativos son altos en todos—, pero tienen más dificultad con "Churn". XGBoost y LightGBM tienen más falsos negativos pero menos falsos positivos, priorizando precisión sobre recall. CatBoost y Random Forest capturan más churns reales a costa de más falsas alarmas.

In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = BASE_DIR / "data" / "processed" / "telco_churn_clean.csv"

df = pd.read_csv(DATA_PATH)

df.head()

El dataset procesado se carga correctamente. `TotalCharges` aparece ahora como `float64`, confirmando que la corrección aplicada en el Notebook 1 está en efecto.

In [ ]:
df.shape

El dataset tiene las mismas 7.043 filas y 21 columnas, sin pérdida de registros durante la limpieza.

In [ ]:
df.info()

`TotalCharges` ya es `float64` en lugar de `object`. La corrección del Notebook 1 está aplicada.

In [ ]:
target = "Churn"

X = df.drop(columns=[target])
y = df[target].map({"No": 0, "Yes": 1})

X.head()

Se separa la variable objetivo `Churn` del resto de features. Se mapea a 0 (No Churn) y 1 (Churn) para que los clasificadores puedan procesarla numéricamente.

In [ ]:
y.value_counts()

5.174 casos de No Churn (0) y 1.869 de Churn (1). El mismo desbalance ~73%/27% del EDA.

In [ ]:
y.value_counts(normalize=True) * 100

El desbalance de 73.46% / 26.54% confirma que se debe usar `class_weight='balanced'` o `scale_pos_weight` según el algoritmo para que el clasificador no ignore la clase minoritaria.

In [ ]:
if "customerID" in X.columns:
    X = X.drop(columns=["customerID"])

X.head()

Se elimina `customerID` y el dataset queda con 19 variables predictoras listas para el pipeline.

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Variables numéricas:")
print(numeric_features)

print("\nVariables categóricas:")
print(categorical_features)

4 variables numéricas y 15 categóricas. Esta separación es necesaria porque el `ColumnTransformer` aplica transformaciones distintas a cada grupo.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Tamaño entrenamiento:", X_train.shape)
print("Tamaño prueba:", X_test.shape)

print("\nDistribución en entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución en prueba:")
print(y_test.value_counts(normalize=True) * 100)

El dataset se divide en 5.634 registros para entrenamiento y 1.409 para prueba. Con `stratify=y` la distribución de churn es prácticamente idéntica en ambos subconjuntos (~73.46%/26.54%), garantizando que la evaluación sea representativa.

### 3. Pipeline de Preprocesamiento

Se construye un `Pipeline` de scikit-learn que encapsula todas las transformaciones de datos
de forma reproducible y evita *data leakage* (filtraciones de información del conjunto de prueba
al de entrenamiento).

**Transformaciones aplicadas:**
- **Variables numéricas** (`SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`):
  1. `SimpleImputer(strategy='median')`: imputa valores faltantes con la mediana, robusta a outliers.
  2. `StandardScaler()`: escala a media 0 y desviación estándar 1.
- **Variables categóricas** (15 variables):
  1. `SimpleImputer(strategy='most_frequent')`: imputa con la categoría más frecuente.
  2. `OneHotEncoder(handle_unknown='ignore')`: codifica en variables binarias, ignorando categorías
     desconocidas en tiempo de inferencia.

El `ColumnTransformer` aplica cada transformación solo a las columnas correspondientes y concatena
los resultados en la matriz de features final.


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

El `ColumnTransformer` aplica a las variables numéricas imputación por mediana y escalado estándar, y a las categóricas imputación por moda y codificación one-hot. Al estar todo dentro del Pipeline, las estadísticas de escalado se calculan solo sobre los datos de entrenamiento y se aplican al conjunto de prueba sin recalcular, evitando data leakage.

In [ ]:
negativos = (y_train == 0).sum()
positivos = (y_train == 1).sum()

scale_pos_weight = negativos / positivos

print("Clientes sin churn:", negativos)
print("Clientes con churn:", positivos)
print("scale_pos_weight:", round(scale_pos_weight, 2))

En el conjunto de entrenamiento hay 4.139 clientes sin churn y 1.495 con churn. El cociente 2.77 es el valor de `scale_pos_weight` que se usará en XGBoost para compensar el desbalance de clases.

### 5. Configuración de Modelos y Búsqueda de Hiperparámetros

Para cada algoritmo se define un espacio de búsqueda de hiperparámetros. La optimización se realiza
con `GridSearchCV`:
- **cv=5:** validación cruzada estratificada de 5 pliegues.
- **scoring='roc_auc':** métrica de selección (robusta al desbalance de clases).
- **n_jobs=-1:** uso de todos los núcleos disponibles para paralelización.
- **refit=True:** re-entrena con los mejores parámetros sobre el conjunto de entrenamiento completo.

**Espacio de búsqueda por modelo:**
- **Random Forest:** n_estimators, max_depth, min_samples_split, min_samples_leaf, class_weight
- **XGBoost:** n_estimators, max_depth, learning_rate, subsample, colsample_bytree, scale_pos_weight
- **CatBoost:** iterations, depth, learning_rate, l2_leaf_reg, auto_class_weights
- **LightGBM:** n_estimators, max_depth, learning_rate, num_leaves, subsample, class_weight


In [ ]:
models_and_params = {
    
    "Random Forest": {
        "model": RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ),
        "params": {
            "classifier__n_estimators": [200, 300],
            "classifier__max_depth": [None, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__min_samples_leaf": [1, 2],
            "classifier__max_features": ["sqrt"],
            "classifier__class_weight": [None, "balanced"]
        }
    },
    
    "XGBoost": {
        "model": XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=-1
        ),
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [3, 5],
            "classifier__learning_rate": [0.05, 0.1],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
            "classifier__scale_pos_weight": [1, scale_pos_weight]
        }
    },
    
    "CatBoost": {
        "model": CatBoostClassifier(
            random_state=42,
            verbose=0,
            allow_writing_files=False
        ),
        "params": {
            "classifier__iterations": [100, 200],
            "classifier__depth": [4, 6],
            "classifier__learning_rate": [0.05, 0.1],
            "classifier__l2_leaf_reg": [3, 5],
            "classifier__auto_class_weights": [None, "Balanced"]
        }
    },
    
    "LightGBM": {
        "model": LGBMClassifier(
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ),
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [-1, 5, 10],
            "classifier__learning_rate": [0.05, 0.1],
            "classifier__num_leaves": [15, 31],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
            "classifier__class_weight": [None, "balanced"]
        }
    }
}

Se definen los espacios de búsqueda para los 4 modelos. En total hay 48 + 64 + 32 + 192 = 336 combinaciones de hiperparámetros, y como cada una se evalúa con 5-fold CV, se entrenan 1.680 modelos en total.

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }
    
    return metrics

La función `evaluate_model()` calcula las 5 métricas principales sobre el conjunto de prueba y devuelve un diccionario. Se llama después de cada GridSearchCV para registrar los resultados comparativos.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []
best_estimators = {}

for model_name, config in models_and_params.items():
    
    print("=" * 80)
    print(f"Entrenando modelo: {model_name}")
    print("=" * 80)
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", config["model"])
        ]
    )
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=config["params"],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    best_estimators[model_name] = best_model
    
    test_metrics = evaluate_model(best_model, X_test, y_test)
    
    result = {
        "model": model_name,
        "best_cv_roc_auc": grid_search.best_score_,
        "test_accuracy": test_metrics["accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1_score": test_metrics["f1_score"],
        "test_roc_auc": test_metrics["roc_auc"],
        "best_params": grid_search.best_params_
    }
    
    results.append(result)
    
    print(f"\nMejor ROC AUC en validación cruzada: {grid_search.best_score_:.4f}")
    print(f"Métricas en prueba:")
    print(test_metrics)
    print(f"\nMejores parámetros:")
    print(grid_search.best_params_)

Los 4 modelos completaron el GridSearchCV. Los AUC-ROC en validación cruzada van de 0.8453 a 0.8504. En el conjunto de prueba XGBoost obtiene el mejor AUC con 0.8481, seguido muy de cerca por CatBoost y LightGBM.

### 6. Comparación de Resultados

Se presenta la tabla comparativa de métricas evaluadas sobre el conjunto de prueba (nunca visto durante
el entrenamiento ni la optimización de hiperparámetros). Las métricas reportadas son:

- **CV AUC:** AUC-ROC promedio en validación cruzada (estimación insesgada del desempeño esperado).
- **Test Accuracy:** proporción de predicciones correctas.
- **Test Precision:** de los clientes predichos como Churn, ¿qué fracción realmente lo es?
- **Test Recall:** de los clientes que realmente hacen Churn, ¿qué fracción predice el modelo?
- **Test F1:** media armónica entre Precision y Recall.
- **Test AUC-ROC:** área bajo la curva ROC (criterio principal de selección).

**Criterio de selección del mejor modelo:** mayor AUC-ROC en el conjunto de prueba.


In [ ]:
results_df = pd.DataFrame(results)

results_df_sorted = results_df.sort_values(
    by="test_roc_auc",
    ascending=False
)

results_df_sorted

XGBoost lidera en AUC-ROC (0.8481) y Accuracy (0.8041). CatBoost tiene el Recall más alto (0.7995) pero menor Precision (0.5164), lo que significa que detecta más churns reales pero con más falsas alarmas. Cada modelo refleja un trade-off distinto entre los dos tipos de error.

In [ ]:
metrics_cols = [
    "model",
    "best_cv_roc_auc",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1_score",
    "test_roc_auc"
]

results_df_sorted[metrics_cols].round(4)

Con los valores redondeados se ve mejor la comparación. La diferencia máxima en AUC entre los 4 modelos es de apenas 0.0089 puntos, lo que indica que las señales predictivas del dataset son fuertes y cualquier algoritmo basado en árboles las captura bien.

In [ ]:
results_plot = results_df_sorted[metrics_cols].melt(
    id_vars="model",
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_plot,
    x="model",
    y="score",
    hue="metric"
)

plt.title("Comparación de métricas por modelo")
plt.xlabel("Modelo")
plt.ylabel("Puntaje")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

El gráfico agrupa las 6 métricas por modelo. XGBoost y LightGBM tienen barras de Accuracy y Precision más altas pero Recall más bajo. CatBoost y Random Forest muestran el patrón inverso: más Recall a costa de menos Precision.

In [ ]:
best_model_name = results_df_sorted.iloc[0]["model"]
best_model = best_estimators[best_model_name]

print("Mejor modelo seleccionado:", best_model_name)

XGBoost es seleccionado como mejor modelo con AUC-ROC de 0.8481 en el conjunto de prueba.

In [ ]:
results_df_sorted.iloc[0]["best_params"]

Los mejores parámetros encontrados son árboles poco profundos (`max_depth=3`), learning rate conservador (0.05), submuestreo de datos y features al 80%, y sin ajuste de peso por clase. La profundidad reducida actúa como regularización y evita el sobreajuste.

In [ ]:
y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    y_pred_best,
    target_names=["No Churn", "Churn"]
))

El modelo predice bien la clase "No Churn" con F1 de 0.87, pero tiene más dificultad con "Churn" (F1=0.58). El Recall de 0.52 para Churn significa que el modelo no detecta aproximadamente la mitad de los clientes que realmente abandonarán.

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
)

disp.plot(cmap="Blues")
plt.title(f"Matriz de confusión - {best_model_name}")
plt.show()

XGBoost clasifica correctamente 942 de 1.035 clientes sin churn (91%) y 193 de 374 con churn (52%). Los 181 falsos negativos —clientes con churn predichos como No Churn— son el error más costoso: representan clientes en riesgo que el sistema no detectará.

In [ ]:
plt.figure(figsize=(8, 6))

for model_name, model in best_estimators.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_proba,
        name=f"{model_name} AUC={auc:.3f}",
        ax=plt.gca()
    )

plt.title("Curvas ROC por modelo")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.show()

Las curvas ROC de los 4 modelos están muy juntas y todas por encima de la diagonal del clasificador aleatorio. XGBoost queda levemente arriba con AUC=0.848.

In [ ]:
REPORTS_DIR = BASE_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

RESULTS_PATH = REPORTS_DIR / "model_results.csv"

results_df_sorted.to_csv(RESULTS_PATH, index=False)

print(f"Resultados guardados en: {RESULTS_PATH}")

Los resultados comparativos de los 4 modelos se guardan en `reports/model_results.csv`.

In [ ]:
preprocessor_fitted = best_model.named_steps["preprocessor"]
classifier_fitted = best_model.named_steps["classifier"]

feature_names = preprocessor_fitted.get_feature_names_out()

feature_names = [
    name.replace("num__", "").replace("cat__", "")
    for name in feature_names
]

if hasattr(classifier_fitted, "feature_importances_"):
    importances = classifier_fitted.feature_importances_
    
    feature_importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values(by="importance", ascending=False)
    
    display(feature_importance_df.head(20))
else:
    print("El modelo seleccionado no tiene atributo feature_importances_.")

`Contract_Month-to-month` domina con 34.7% de importancia, más que las siguientes 5 variables sumadas. `TechSupport_No` y `OnlineSecurity_No` ocupan el segundo y tercer lugar, consistente con los patrones del EDA.

In [ ]:
if hasattr(classifier_fitted, "feature_importances_"):
    top_features = feature_importance_df.head(20)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(
        data=top_features,
        x="importance",
        y="feature"
    )
    
    plt.title(f"Top 20 variables más importantes - {best_model_name}")
    plt.xlabel("Importancia")
    plt.ylabel("Variable")
    plt.tight_layout()
    plt.show()

`Contract_Month-to-month` domina ampliamente en la parte derecha del gráfico. Las demás variables importantes son binarias derivadas del one-hot encoding. `tenure` es la única variable numérica que aparece con importancia relevante en el top 20.

In [ ]:
# Importancia de características para los 4 modelos (subplots comparativos)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

model_order = ["Random Forest", "XGBoost", "CatBoost", "LightGBM"]
colors_fi = {
    "Random Forest": "#e74c3c",
    "XGBoost": "#27ae60",
    "CatBoost": "#f39c12",
    "LightGBM": "#8e44ad"
}
N_TOP = 12

for i, model_name in enumerate(model_order):
    if model_name not in best_estimators:
        continue
    m = best_estimators[model_name]
    clf = m.named_steps["classifier"]
    prep = m.named_steps["preprocessor"]
    feat_names_raw = prep.get_feature_names_out()
    feat_names = [n.replace("num__", "").replace("cat__", "") for n in feat_names_raw]

    if not hasattr(clf, "feature_importances_"):
        axes[i].text(0.5, 0.5, "No disponible", ha="center", va="center")
        axes[i].set_title(model_name)
        continue

    df_fi = pd.DataFrame({
        "feature": feat_names,
        "importance": clf.feature_importances_
    }).sort_values("importance", ascending=False).head(N_TOP)

    df_fi_sorted = df_fi.sort_values("importance", ascending=True)

    axes[i].barh(
        df_fi_sorted["feature"], df_fi_sorted["importance"],
        color=colors_fi[model_name], edgecolor="white", alpha=0.85
    )
    axes[i].set_title(f"{model_name} — Top {N_TOP} Features", fontweight="bold", fontsize=12)
    axes[i].set_xlabel("Importancia", fontsize=10)
    axes[i].tick_params(axis="y", labelsize=9)
    axes[i].grid(axis="x", alpha=0.3)

plt.suptitle(
    "Importancia de Características por Modelo\n(Features más relevantes para la predicción de Churn)",
    fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()


En los cuatro modelos aparecen las mismas variables en los primeros puestos. `Contract_Month-to-month` domina en todos —especialmente en XGBoost donde llega al 35%—, seguida de `TechSupport_No`, `OnlineSecurity_No` e `InternetService_Fiber optic`. `tenure` es la única variable numérica que aparece de forma consistente entre las más importantes. Que cuatro algoritmos distintos coincidan en las mismas variables le da mucha solidez a esas conclusiones.

In [ ]:
if hasattr(classifier_fitted, "feature_importances_"):
    FEATURE_IMPORTANCE_PATH = REPORTS_DIR / "feature_importance.csv"
    
    feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)
    
    print(f"Importancia de características guardada en: {FEATURE_IMPORTANCE_PATH}")

La importancia de características se guarda en `reports/feature_importance.csv`.

In [ ]:
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODELS_DIR / "best_model.joblib"

joblib.dump(best_model, MODEL_PATH)

print(f"Modelo guardado en: {MODEL_PATH}")

El pipeline completo (preprocesador + XGBoost) se serializa en `models/best_model.joblib`. Al guardar el pipeline entero —no solo el clasificador—, el preprocesamiento se aplica automáticamente en cada inferencia sin necesidad de pasos adicionales.

In [ ]:
metadata = {
    "best_model_name": best_model_name,
    "target": target,
    "target_mapping": {
        "No": 0,
        "Yes": 1
    },
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "features_used": X.columns.tolist(),
    "selection_metric": "test_roc_auc"
}

METADATA_PATH = MODELS_DIR / "model_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print(f"Metadatos guardados en: {METADATA_PATH}")

Los metadatos del modelo (nombre, features, mapping de la variable objetivo, métrica de selección) se guardan en JSON. Son leídos por el Notebook 3 y por la API para garantizar que se usen exactamente las mismas features que durante el entrenamiento.

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)[0]
prediction_proba = loaded_model.predict_proba(sample)[0, 1]

print("Predicción:", prediction)
print("Probabilidad de churn:", round(prediction_proba, 4))

El modelo cargado funciona correctamente: para el primer cliente del conjunto de prueba predice No Churn con una probabilidad de churn de apenas 3.44%.

### 7. Conclusiones del Modelado Predictivo

El proceso de modelado predictivo con validación cruzada y búsqueda de hiperparámetros permite
establecer las siguientes conclusiones:

1. **Modelo seleccionado:** XGBoost con `max_depth=3`, `learning_rate=0.05`, `n_estimators=100`,
   `subsample=0.8`, `colsample_bytree=0.8`. Su AUC-ROC de 0.8481 en el conjunto de prueba representa
   un desempeño sólido para este tipo de problema de clasificación binaria con desbalance moderado.

2. **Homogeneidad del espacio de soluciones:** Los cuatro algoritmos presentan AUC-ROC muy similares
   (diferencia máxima de 0.009), lo que sugiere que el dataset contiene señales predictivas fuertes
   y claras que cualquier algoritmo basado en árboles puede capturar eficientemente.

3. **Robustez del pipeline:** La integración de preprocesamiento, optimización y clasificación en un
   único `Pipeline` de scikit-learn garantiza que las transformaciones se apliquen correctamente
   tanto en entrenamiento como en inferencia, eliminando el riesgo de data leakage.

4. **Variables críticas identificadas:** `Contract_Month-to-month`, `TechSupport_No`,
   `OnlineSecurity_No`, `InternetService_Fiber optic` y `tenure` son las variables que más
   guían las predicciones del modelo, en coherencia con los hallazgos del EDA.

5. **Modelo serializado:** El pipeline completo (preprocesamiento + clasificador) fue guardado en
   `models/best_model.joblib` para su uso en el análisis de interpretabilidad (Notebook 3) y en
   la API de predicción en tiempo real (FastAPI).
